In [0]:
# Bridge provisória CINE–CBO (versão aproximada, documentada)

Constrói `workspace.gold.bridge_cine_cbo_provisoria`, mapeando **áreas específicas CINE Brasil (3 dígitos)** a **subgrupos principais CBO 2002 (3 dígitos)**, enquanto o CSV oficial de correspondência do MEC não chega ao volume `/Volumes/workspace/bronze/raw_cine_cbo`.

## Decisões de método

1. **Fonte alternativa** — mapeamento curado pela equipe (dicionário explícito neste notebook), ancorado na estrutura hierárquica CINE 2013 × CBO 2002. **Não é correspondência oficial.**
2. **Flag de mapeamento aproximado** — toda linha recebe `tipo_mapeamento = 'APROXIMADO'` e `aprovado_oficial = false`. A tabela oficial (`bridge_cine_cbo`, notebook 033) continua reservada para o CSV do MEC — esta é uma tabela **separada** para não poluir o slot oficial.
3. **Grão** — área específica CINE × subgrupo principal CBO. É o grão consumível pelas features do modelo (UF × grupo ocupacional) e é o máximo defensável sem a fonte oficial: corresponder rótulo CINE a ocupação CBO (6 dígitos) sem o arquivo do MEC geraria correspondências fictícias.
4. **Rastreabilidade** — cada par carrega `confianca` (ALTA/MEDIA/BAIXA) e `justificativa` textual; auditoria de cobertura gravada em `workspace.gold.bridge_cine_cbo_provisoria_cobertura`.
5. **Áreas sem ancoragem defensável** (interdisciplinares, "em processo de definição", serviços pessoais de baixa barreira) ficam **fora da bridge** e aparecem no relatório de cobertura com o motivo — não são mapeadas por forçação.

In [0]:
from pyspark.sql import functions as F
from datetime import datetime

# Parâmetros (widgets) — sobrescritos por jobs/agendamentos sem alterar o código
dbutils.widgets.text("versao_mapeamento", "0.1.0-provisoria", "Versão do mapeamento provisório")
dbutils.widgets.text("versao_transformacao", "0.1.0", "Versão da transformação")
dbutils.widgets.text("tabela_bridge", "workspace.gold.bridge_cine_cbo_provisoria", "Tabela-alvo da bridge")
dbutils.widgets.text("tabela_cobertura", "workspace.gold.bridge_cine_cbo_provisoria_cobertura", "Tabela-alvo da auditoria de cobertura")

VERSAO_MAPEAMENTO = dbutils.widgets.get("versao_mapeamento")
VERSAO_TRANSFORMACAO = dbutils.widgets.get("versao_transformacao")
BRIDGE = dbutils.widgets.get("tabela_bridge")
COBERTURA = dbutils.widgets.get("tabela_cobertura")

DIM_CINE = "workspace.gold.dim_cine_brasil"
DIM_CBO = "workspace.gold.dim_cbo_ocupacao"
DT_PROCESSAMENTO = datetime.now()

print(f"Versão do mapeamento: {VERSAO_MAPEAMENTO}")
print(f"Tabela-alvo: {BRIDGE}")
print(f"Auditoria de cobertura: {COBERTURA}")

In [0]:
# Catálogo de áreas específicas CINE presentes na dimensão (chave do lado CINE)
df_cine_areas = (
    spark.table(DIM_CINE)
    .select(
        "codigo_cine_area_geral", "nome_cine_area_geral",
        "codigo_cine_area_especifica", "nome_cine_area_especifica"
    )
    .distinct()
)

# Catálogo de subgrupos principais CBO (3 dígitos) presentes na dimensão (chave do lado CBO)
df_cbo_subgrupos = (
    spark.table(DIM_CBO)
    .select(
        F.substring("codigo_cbo", 1, 3).alias("codigo_cbo_subgrupo"),
        "nome_ocupacao"
    )
    .groupBy("codigo_cbo_subgrupo")
    .agg(F.min("nome_ocupacao").alias("exemplo_ocupacao"), F.count("*").alias("qtd_ocupacoes"))
)

qtd_areas = df_cine_areas.count()
qtd_subgrupos = df_cbo_subgrupos.count()
print(f"Áreas específicas CINE na dimensão: {qtd_areas}")
print(f"Subgrupos principais CBO na dimensão: {qtd_subgrupos}")

In [0]:
# MAPEAMENTO CURADO PELA EQUIPE — FONTE ALTERNATIVA (não oficial)
# Estrutura: {codigo_cine_area_especifica: [(codigo_cbo_subgrupo, confianca, justificativa), ...]}
# Âncoras: 212 = profissionais de TIC | 223 = profissionais de saúde nível superior
#          23x = professores e coordenação | 24x = jurídicas | 214 = engenheiros/arquitetos
#          222 = agrônomos | 221 = biólogos | 213 = físicos/químicos | 211 = atuários
#          251 = ciências sociais | 252 = administradores | 261 = comunicação/informação | 262 = artistas

MAPEAMENTO = {
    "011": [("231", "ALTA", "Licenciaturas formam professores da educação básica e coordenação pedagógica (CBO 231/232/233/234/239)."),
            ("232", "ALTA", "Professores de artes do ensino médio vêm das licenciaturas em artes."),
            ("233", "ALTA", "Instrutores/técnicos de ensino agropecuário formados por licenciaturas e bacharelados de educação."),
            ("234", "ALTA", "Professores de nível superior e do ensino médio vêm de programas de educação."),
            ("239", "ALTA", "Coordenadores pedagógicos e especialistas de ensino são a saída natural dos programas de educação.")],
    "021": [("262", "MEDIA", "Bacharelados e licenciaturas em artes alimentam artistas (CBO 262) e profissões audiovisuais."),
            ("261", "BAIXA", "Parte dos programas de artes/design atinge profissionais de comunicação e informação (261).")],
    "022": [("251", "BAIXA", "Filosofia e ciências humanas alimentam antropólogos/filósofos/sociólogos (251); correspondência difusa."),
            ("234", "BAIXA", "Parte dos formandos em humanidades leciona no ensino superior/médio (234).")],
    "023": [("261", "MEDIA", "Letras alimentam jornalistas, tradutores, bibliotecários e linguistas (CBO 261).")],
    "031": [("251", "MEDIA", "Ciências sociais e comportamentais formam sociólogos, antropólogos, psicólogos e pesquisadores (251).")],
    "032": [("261", "MEDIA", "Comunicação social e biblioteconomia formam jornalistas, publicitários e bibliotecários (261)."),
            ("253", "MEDIA", "Publicidade e propaganda alimentam agenciadores de propaganda e profissionais de marketing (253).")],
    "041": [("252", "ALTA", "Administração, ciências contábeis e econômicas formam administradores e afins (CBO 252)."),
            ("211", "MEDIA", "Finanças, atuária e contabilidade também abastecem atuários (211).")],
    "042": [("241", "ALTA", "Direito forma advogados (241) — correspondência direta (98 mil formandos × 23 mil admissões no painel)."),
            ("242", "MEDIA", "Direito também abastece defensores, promotores e juízes (242).")],
    "051": [("221", "MEDIA", "Ciências biológicas formam biólogos (221) e profissionais de biotecnologia."),
            ("203", "MEDIA", "Parte dos biólogos atua como pesquisadores (203).")],
    "052": [("221", "BAIXA", "Meio ambiente alimenta biólogos (221) e engenheiros ambientais; ancoragem parcial."),
            ("222", "BAIXA", "Engenharias florestais/ambientais são capturadas em 222 (engenheiros agronômicos).")],
    "053": [("213", "MEDIA", "Física, química e astronomia formam profissionais das ciências físicas (CBO 213).")],
    "054": [("211", "MEDIA", "Matemática e estatística formam atuários, estatísticos e matemáticos (CBO 211).")],
    "061": [("212", "ALTA", "Computação e TIC formam profissionais de TIC (CBO 212) — âncora do painel (62 mil formandos × 140 mil admissões)."),
            ("211", "BAIXA", "Ciência de dados e computação aplicada também atinge estatísticos/atuários (211).")],
    "068": [("212", "BAIXA", "Interdisciplinares com TIC (mídia digital, sistemas embarcados) majoritariamente em 212; confiança menor.")],
    "071": [("214", "MEDIA", "Engenharias formam engenheiros (CBO 214, família ampla de engenharias)."),
            ("202", "MEDIA", "Engenharias integradas/mecatrônicas/afins ficam em 202 (profissionais de engenharias integradas).")],
    "072": [("214", "BAIXA", "Engenharia de produção/processos alimentam engenheiros industriais (214); ancoragem parcial."),
            ("202", "BAIXA", "Parte alcança profissionais de engenharias integradas (202).")],
    "073": [("214", "ALTA", "Arquitetura e urbanismo e engenharia civil formam arquitetos e engenheiros civis (CBO 214).")],
    "081": [("222", "ALTA", "Agronomia forma engenheiros agrônomos (CBO 222).")],
    "082": [("222", "MEDIA", "Engenharia florestal é capturada em 222 (engenheiros agronômicos/florestais).")],
    "083": [("222", "BAIXA", "Engenharia de pesca e aquicultura têm ancoragem fraca; provisoriamente em 222."),
            ("221", "BAIXA", "Biologia marinha alcança biólogos (221).")],
    "084": [("221", "BAIXA", "Medicina veterinária não tem família própria visível sem a fonte oficial; provisoriamente em 221 (biológicas) — CONFIRMAR com o CSV do MEC.")],
    "091": [("223", "ALTA", "Medicina, enfermagem, farmácia, odontologia, fisioterapia e afins formam os profissionais de saúde nível superior (CBO 223, 132 ocupações) — âncora do painel (54,5 mil formandos × 209 mil admissões)."),
            ("225", "MEDIA", "Parte da saúde (medicina integrativa/terapias) alcança 225."),
            ("226", "MEDIA", "Fonoaudiologia, terapia ocupacional e afins alcançam 226.")],
    "092": [("224", "MEDIA", "Educação física e bem-estar formam profissionais de educação física (CBO 224)."),
            ("226", "MEDIA", "Psicologia e terapias alcançam 226 (arteterapeutas e afins)."),
            ("251", "BAIXA", "Psicologia também está representada nas ciências sociais aplicadas (251).")],
    "103": [("351", "BAIXA", "Segurança pública/nacional alcança agentes de inteligência e fiscalização (351/352); ancoragem fraca."),
            ("352", "BAIXA", "Idem, fiscalização e segurança de nível técnico-superior (352).")],
}

# Áreas CINE deixadas FORA da bridge, com motivo documentado
SEM_MAPEAMENTO = {
    "001": "Programas básicos — não correspondem a formação de nível superior específica.",
    "018": "Interdisciplinar em educação — sem ancoragem ocupacional identificável.",
    "019": "Educação em processo de definição — sem ancoragem.",
    "028": "Interdisciplinar em artes e humanidades — sem ancoragem.",
    "038": "Interdisciplinar em ciências sociais — sem ancoragem.",
    "039": "Ciências sociais em processo de definição — sem ancoragem.",
    "048": "Interdisciplinar em negócios/direito — sem ancoragem.",
    "049": "Negócios/direito em processo de definição — sem ancoragem.",
    "058": "Interdisciplinar em ciências naturais — sem ancoragem.",
    "078": "Interdisciplinar em engenharia — sem ancoragem.",
    "088": "Interdisciplinar em agricultura/veterinária — sem ancoragem.",
    "098": "Interdisciplinar em saúde/bem-estar — sem ancoragem.",
    "108": "Interdisciplinar em serviços — sem ancoragem ocupacional identificável.",
    "101": "Serviços pessoais (estética, hotelaria) — demanda de baixa barreira; sem família CBO de nível superior defensável sem a fonte oficial.",
    "102": "Higiene e saúde ocupacional — formação de nível TÉCNICO, fora do escopo superior do Censo INEP na gold.",
    "104": "Serviços de transporte — sem família CBO de nível superior identificável sem a fonte oficial.",
}

df_map = spark.createDataFrame(
    [(cine, cbo, conf, just) for cine, pares in MAPEAMENTO.items() for (cbo, conf, just) in pares],
    "codigo_cine_area_especifica STRING, codigo_cbo_subgrupo STRING, confianca STRING, justificativa STRING"
)
print(f"Pares CINE área × CBO subgrupo definidos: {df_map.count()}")
print(f"Áreas CINE com mapeamento: {df_map.select('codigo_cine_area_especifica').distinct().count()} de {qtd_areas}")
print(f"Áreas fora da bridge (documentadas): {len(SEM_MAPEAMENTO)}")

In [0]:
# Validação 1 — todas as chaves CINE existem na dimensão
cines_validos = set(r["codigo_cine_area_especifica"] for r in df_cine_areas.collect())
cines_no_map = set(r["codigo_cine_area_especifica"] for r in df_map.select("codigo_cine_area_especifica").distinct().collect())
invalidos_cine = cines_no_map - cines_validos

# Validação 2 — todos os subgrupos CBO existem na dimensão
subgrupos_validos = set(r["codigo_cbo_subgrupo"] for r in df_cbo_subgrupos.collect())
subgrupos_no_map = set(r["codigo_cbo_subgrupo"] for r in df_map.select("codigo_cbo_subgrupo").distinct().collect())
invalidos_cbo = subgrupos_no_map - subgrupos_validos

# Validação 3 — confiança só nos valores permitidos
conf_invalidas = [r["confianca"] for r in df_map.select("confianca").distinct().collect() if r["confianca"] not in ("ALTA", "MEDIA", "BAIXA")]

# Validação 4 — áreas do dicionário SEM_MAPEAMENTO também devem existir na dimensão
invalidos_sem_map = set(SEM_MAPEAMENTO.keys()) - cines_validos

if invalidos_cine or invalidos_cbo or conf_invalidas or invalidos_sem_map:
    raise ValueError(
        f"VALIDAÇÃO FALHOU — abortando sem gravar nada.\n"
        f"CINE inexistentes na dimensão: {sorted(invalidos_cine)}\n"
        f"CBO inexistentes na dimensão: {sorted(invalidos_cbo)}\n"
        f"Confiança inválida: {conf_invalidas}\n"
        f"Áreas SEM_MAPEAMENTO inexistentes na dimensão: {sorted(invalidos_sem_map)}"
    )

# Cobertura total: toda área da dimensão deve estar em MAPEAMENTO ou SEM_MAPEAMENTO
nao_cobertas = cines_validos - cines_no_map - set(SEM_MAPEAMENTO.keys())
if nao_cobertas:
    print(f"ATENÇÃO — áreas CINE sem posição declarada (nem mapeadas nem justificadas): {sorted(nao_cobertas)}")
else:
    print("Cobertura declarada: 100% das áreas CINE da dimensão estão em MAPEAMENTO ou SEM_MAPEAMENTO.")

print("Validações de chaves e valores: OK.")

In [0]:
# Construção da bridge: junta o mapeamento curado aos nomes oficiais das dimensões
df_bridge = (
    df_map
    .join(df_cine_areas.select(
        "codigo_cine_area_especifica", "nome_cine_area_especifica",
        "codigo_cine_area_geral", "nome_cine_area_geral"
    ), "codigo_cine_area_especifica", "left")
    .join(df_cbo_subgrupos.select("codigo_cbo_subgrupo", "exemplo_ocupacao"), "codigo_cbo_subgrupo", "left")
    .select(
        "codigo_cine_area_geral", "nome_cine_area_geral",
        "codigo_cine_area_especifica", "nome_cine_area_especifica",
        "codigo_cbo_subgrupo",
        F.substring("codigo_cbo_subgrupo", 1, 1).alias("codigo_cbo_grande_grupo"),
        "exemplo_ocupacao",
        F.lit("APROXIMADO").alias("tipo_mapeamento"),
        F.lit("AREA_ESPECIFICA_X_SUBGRUPO_PRINCIPAL").alias("nivel_mapeamento"),
        "confianca",
        "justificativa",
        F.lit("mapeamento_curado_equipe_v0.1 (sem CSV oficial MEC)").alias("fonte_mapeamento"),
        F.lit(False).alias("aprovado_oficial"),
        F.lit(VERSAO_MAPEAMENTO).alias("versao_mapeamento"),
        F.lit(DT_PROCESSAMENTO).alias("dt_processamento"),
        F.lit(VERSAO_TRANSFORMACAO).alias("versao_transformacao"),
    )
    .orderBy("codigo_cine_area_especifica", "codigo_cbo_subgrupo")
)

print(f"Linhas da bridge provisória: {df_bridge.count()}")
print("Distribuição por confiança:")
display(df_bridge.groupBy("confianca").count().orderBy("confianca"))

In [0]:
DDL_BRIDGE = f"""
CREATE TABLE IF NOT EXISTS {BRIDGE} (
    codigo_cine_area_geral STRING COMMENT 'Código CINE da área geral (contexto)',
    nome_cine_area_geral STRING COMMENT 'Nome CINE da área geral (contexto)',
    codigo_cine_area_especifica STRING COMMENT 'Código CINE da área específica (chave do lado CINE, 3 dígitos)',
    nome_cine_area_especifica STRING COMMENT 'Nome CINE da área específica',
    codigo_cbo_subgrupo STRING COMMENT 'Código CBO do subgrupo principal (chave do lado CBO, 3 dígitos)',
    codigo_cbo_grande_grupo STRING COMMENT 'Código CBO do grande grupo (1º dígito, contexto)',
    exemplo_ocupacao STRING COMMENT 'Ocupação representativa do subgrupo (apenas ilustrativa)',
    tipo_mapeamento STRING COMMENT 'Sempre APROXIMADO nesta tabela — versão provisória sem fonte oficial',
    nivel_mapeamento STRING COMMENT 'Grão do par: AREA_ESPECIFICA_X_SUBGRUPO_PRINCIPAL',
    confianca STRING COMMENT 'Confiança do par: ALTA, MEDIA ou BAIXA',
    justificativa STRING COMMENT 'Justificativa textual do par, para auditoria',
    fonte_mapeamento STRING COMMENT 'Fonte do mapeamento (curada pela equipe, não oficial)',
    aprovado_oficial BOOLEAN COMMENT 'Sempre false — aguarda correspondência oficial do MEC',
    versao_mapeamento STRING COMMENT 'Versão do mapeamento provisório',
    dt_processamento TIMESTAMP COMMENT 'Data e hora da gravação',
    versao_transformacao STRING COMMENT 'Versão da transformação'
)
"""

spark.sql(DDL_BRIDGE)

# Gravação idempotente: só substitui se a versão do mapeamento mudou
ja_existe = spark.sql(f"SELECT versao_mapeamento FROM {BRIDGE}").collect()

if not ja_existe:
    df_bridge.write.mode("append").saveAsTable(BRIDGE)
    print(f"Bridge criada e gravada: {BRIDGE}")
elif any(r["versao_mapeamento"] == VERSAO_MAPEAMENTO for r in ja_existe):
    print(f"Versão {VERSAO_MAPEAMENTO} já gravada — nada a fazer (idempotente).")
else:
    # Nova versão do mapeamento: substitui o conteúdo integralmente
    df_bridge.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(BRIDGE)
    print(f"Bridge substituída pela versão {VERSAO_MAPEAMENTO}.")

In [0]:
# Auditoria de cobertura: mostra o que ficou fora e por quê
df_todas_areas = df_cine_areas.select("codigo_cine_area_especifica", "nome_cine_area_especifica").distinct()

df_cobertura = (
    df_todas_areas
    .join(df_map.select("codigo_cine_area_especifica").withColumn("mapeada", F.lit(True)).distinct(), "codigo_cine_area_especifica", "left")
    .withColumn("mapeada", F.coalesce(F.col("mapeada"), F.lit(False)))
    .withColumn("posicao", F.when(F.col("mapeada"), F.lit("MAPEADA (APROXIMADO)"))
                          .otherwise(F.lit("FORA DA BRIDGE")))
    .withColumn("motivo_fora", F.udf(lambda c: SEM_MAPEAMENTO.get(c))("codigo_cine_area_especifica"))
    .join(df_bridge.groupBy("codigo_cine_area_especifica").agg(F.count("*").alias("qtd_pares_cbo")), "codigo_cine_area_especifica", "left")
    .withColumn("versao_mapeamento", F.lit(VERSAO_MAPEAMENTO))
    .withColumn("dt_processamento", F.lit(DT_PROCESSAMENTO))
    .select("codigo_cine_area_especifica", "nome_cine_area_especifica", "posicao", "qtd_pares_cbo", "motivo_fora", "versao_mapeamento", "dt_processamento")
    .orderBy(F.desc("mapeada"), "codigo_cine_area_especifica")
)

# Gravação idempotente em APPEND — a cobertura é um log versionado de auditoria:
# cada versao_mapeamento anexa seu snapshot uma única vez; o consumo filtra pela versão vigente.
# (Substitui o padrão DROP+overwrite, que é destrutivo e não passaria em revisão de pipeline.)
versoes_ja_gravadas = []
try:
    versoes_ja_gravadas = [r["versao_mapeamento"] for r in spark.sql(f"SELECT DISTINCT versao_mapeamento FROM {COBERTURA}").collect()]
except Exception:
    pass  # tabela ainda não existe — primeira gravação

if VERSAO_MAPEAMENTO in versoes_ja_gravadas:
    print(f"Cobertura da versão {VERSAO_MAPEAMENTO} já gravada — nada a fazer (idempotente).")
else:
    df_cobertura.write.mode("append").saveAsTable(COBERTURA)
    print(f"Auditoria de cobertura gravada (append): {COBERTURA}")

resumo = df_cobertura.groupBy("posicao").count().collect()
for r in resumo:
    print(f"  {r['posicao']}: {r['count']} áreas")
print("\nResumo da bridge gravada:")
display(spark.sql(f"SELECT confianca, COUNT(*) AS pares, COUNT(DISTINCT codigo_cine_area_especifica) AS areas, COUNT(DISTINCT codigo_cbo_subgrupo) AS subgrupos FROM {BRIDGE} GROUP BY confianca ORDER BY confianca"))

## Observações

* **Consumo pelo modelo** — para converter formandos CINE em formandos por grupo ocupacional: agregue o Censo por UF × `codigo_cine_area_especifica`, junte esta bridge na área específica e reparta a contagem da área entre os subgrupos CBO do par (uniforme, ou ponderada por `confianca` — decisão de feature engineering documentada depois na EDA).
* **Substituição pela fonte oficial** — quando o CSV do MEC chegar a `/Volumes/workspace/bronze/raw_cine_cbo`, o notebook 033 materializa `bridge_cine_cbo`; esta tabela provisória deve então ser descontinuada (mantida como histórico pela coluna `versao_mapeamento`).
* **Pendências de confiança BAIXA** — pares marcados para revisão com a fonte oficial: veterinária (084→221), pesca (083), artes (021→261), humanidades (022), interdisciplinares TIC (068) e produção (072).